> **ARCHIVED — do not use to reproduce results.** This is the original exploratory notebook; its saved outputs are kept only as a record of what the report was based on.
> Known problems (all fixed in the replacement): the form writes `/tmp/input_protein_ligand.csv` but inference reads `data/protein_ligand_example.csv`; upstream is cloned unpinned (the saved ESM-preparation call fails with `unrecognized arguments: --protein_ligand_csv`; the saved inference output ends in a `KeyboardInterrupt`); results are exported from a different folder than the one inference writes to; the viewer shows hard-coded receptors, not predicted poses; whole-PDBBind preparation is mixed into the single-example flow.
> 
> **Use instead:** [`notebooks/02_docking_colab.ipynb`](notebooks/02_docking_colab.ipynb) (single complex, pinned, verified output, pose viewer) and [`notebooks/03_pdbbind_benchmark_prep.ipynb`](notebooks/03_pdbbind_benchmark_prep.ipynb) (benchmark data).


This is the first version of the notebook. Use this newer version instead: https://colab.research.google.com/drive/1CTtUGg05-2MtlWmfJhqzLTtkDDaxCDOQ

# DiffDock
Dock a PDB files and a SMILES with [DiffDock](https://github.com/gcorso/DiffDock).

Select Runtime / Run all to run an example PDB file and SMILES.

May require "premium GPU" (colab pro), and even then it may fail on large complexes.

In [ ]:
#@title PDB + SMILES input

PDB_id = '' #@param {type:"string"}
SMILES_or_pubchem_id = '' #@param {type:"string"}

#@markdown Download a tar file containing all results?
download_results = True #@param {type:"boolean"}

In [ ]:
import os
import requests
import time
from random import random

def download_pdb_file(pdb_id: str) -> str:
    """Download pdb file as a string from rcsb.org"""
    PDB_DIR ="/tmp/pdb/"
    os.makedirs(PDB_DIR, exist_ok=True)

    # url or pdb_id
    if pdb_id.startswith('http'):
        url = pdb_id
        filename = url.split('/')[-1]
    else:
        url = f"http://files.rcsb.org/view/{pdb_id}.pdb"
        filename = f'{pdb_id}.pdb'

    cache_path = os.path.join(PDB_DIR, filename)
    if os.path.exists(cache_path):
        return cache_path

    pdb_req = requests.get(url)
    pdb_req.raise_for_status()
    open(cache_path, 'w').write(pdb_req.text)
    return cache_path

def download_smiles_str(pubchem_id: str, retries:int = 2) -> str:
    """Given a pubchem id, get a smiles string"""
    while True:
        req = requests.get(f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/CID/{pubchem_id}/property/CanonicalSMILES/CSV")
        smiles_url_csv = req.text if req.status_code == 200 else None
        if smiles_url_csv is not None:
            break
        if retries == 0:
            return None
        time.sleep(1+random())
        retries -= 1
    retour = smiles_url_csv.splitlines()[1].split(',')[1].strip('"').strip("'") if smiles_url_csv is not None else None
    print(retour)
    return smiles_url_csv.splitlines()[1].split(',')[1].strip('"').strip("'") if smiles_url_csv is not None else None

In [ ]:
if not PDB_id or not SMILES_or_pubchem_id:
    PDB_id = "6agt"
    SMILES_or_pubchem_id = "COc(cc1)ccc1C#N"
    print(f"No input supplied. Using example data: {PDB_id} and {SMILES_or_pubchem_id}")

# to run many PDB+smiles at once, fill in a list of PDB_files and smiles here...
pdb_files = [download_pdb_file(PDB_id)]
smiless = [download_smiles_str(SMILES_or_pubchem_id) if str(SMILES_or_pubchem_id).isnumeric() else SMILES_or_pubchem_id]
print(smiless)
with open("/tmp/input_protein_ligand.csv", 'w') as out:
    out.write("protein_path,ligand\n")
    for pdb_file in pdb_files:
        for smiles in smiless:
            out.write(f"{pdb_file},{smiles}\n")

No input supplied. Using example data: 6agt and COc(cc1)ccc1C#N
['COc(cc1)ccc1C#N']


In [ ]:
print(download_smiles_str(SMILES_or_pubchem_id))

None


## Install prerequisites

In [ ]:
!pip install ipython-autotime
%load_ext autotime

time: 692 µs (started: 2024-12-03 21:52:10 +00:00)


In [ ]:
if not os.path.exists("/content/DiffDock"):
    %cd /content
    !git clone https://github.com/gcorso/DiffDock.git
    %cd /content/DiffDock
    #!git checkout 0f9c419 # remove/update for more up to date code

time: 1.1 ms (started: 2024-12-03 21:52:10 +00:00)


In [ ]:
''' !pip install pyg==0.7.1 --quiet
!pip install pyyaml==6.0 --quiet
!pip install scipy==1.7.3 --quiet
!pip install networkx==2.6.3 --quiet
!pip install biopython==1.79 --quiet
!pip install rdkit-pypi==2022.03.5 --quiet
!pip install e3nn==0.5.0 --quiet
!pip install spyrmsd==0.5.2 --quiet
!pip install pandas==1.3.5 --quiet
!pip install biopandas==0.4.1 --quiet
!pip install torch==1.12.1+cu113 --quiet '''
!pip install rdkit-pypi
!pip install biopython
!pip install biopandas
!pip install e3nn
!pip install spyrmsd
!pip install networkx
!pip install prody

time: 51.4 s (started: 2024-12-03 21:52:10 +00:00)


In [ ]:
import torch

try:
    import torch_geometric
except ModuleNotFoundError:
    !pip uninstall torch-scatter torch-sparse torch-geometric torch-cluster  --y
    !pip install torch-scatter -f https://data.pyg.org/whl/torch-{torch.__version__}.html --quiet
    !pip install torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html --quiet
    !pip install torch-cluster -f https://data.pyg.org/whl/torch-{torch.__version__}.html --quiet
    !pip install git+https://github.com/pyg-team/pytorch_geometric.git  --quiet # no version for some reason??

time: 900 µs (started: 2024-12-03 21:53:02 +00:00)


### Download 2GB PDBBind dataset
unnecessary for inference

In [ ]:
#!test -d /content/DiffDock/data/PDBBind_processed || (wget https://zenodo.org/record/6034088/files/PDBBind.zip && unzip -q PDBBind.zip && mv PDBBind_processed /content/DiffDock/data/)

time: 2.89 ms (started: 2024-12-03 21:53:02 +00:00)


## Install ESM and prepare PDB file for ESM

In [ ]:
if not os.path.exists("/content/DiffDock/esm"):
    %cd /content/DiffDock
    !git clone https://github.com/facebookresearch/esm
    %cd /content/DiffDock/esm
    #!git checkout f07aed6 # remove/update for more up to date code
    !sudo pip install -e .
    %cd /content/DiffDock

time: 807 µs (started: 2024-12-03 21:53:02 +00:00)


In [ ]:
%cd /content/DiffDock
!python datasets/esm_embedding_preparation.py --protein_ligand_csv /tmp/input_protein_ligand.csv --out_file data/prepared_for_esm.fasta

/content/DiffDock
usage: esm_embedding_preparation.py [-h] [--out_file OUT_FILE] [--dataset DATASET]
                                    [--data_dir DATA_DIR]
esm_embedding_preparation.py: error: unrecognized arguments: --protein_ligand_csv /tmp/input_protein_ligand.csv
time: 510 ms (started: 2024-12-03 22:27:31 +00:00)


In [ ]:
%cd /content/DiffDock
%env HOME=esm/model_weights
%env PYTHONPATH=$PYTHONPATH:/content/DiffDock/esm
!python /content/DiffDock/esm/scripts/extract.py esm2_t33_650M_UR50D data/prepared_for_esm.fasta data/esm2_output --repr_layers 33 --include per_tok

/content/DiffDock
env: HOME=esm/model_weights
env: PYTHONPATH=$PYTHONPATH:/content/DiffDock/esm
Traceback (most recent call last):
  File "/content/DiffDock/esm/scripts/extract.py", line 140, in <module>
    main()
  File "/content/DiffDock/esm/scripts/extract.py", line 137, in main
    run(args)
  File "/content/DiffDock/esm/scripts/extract.py", line 71, in run
    model = model.cuda()
  File "/usr/local/lib/python3.10/dist-packages/torch/nn/modules/module.py", line 1050, in cuda
    return self._apply(lambda t: t.cuda(device))
  File "/usr/local/lib/python3.10/dist-packages/torch/nn/modules/module.py", line 900, in _apply
    module._apply(fn)
  File "/usr/local/lib/python3.10/dist-packages/torch/nn/modules/module.py", line 927, in _apply
    param_applied = fn(param)
  File "/usr/local/lib/python3.10/dist-packages/torch/nn/modules/module.py", line 1050, in <lambda>
    return self._apply(lambda t: t.cuda(device))
KeyboardInterrupt
^C
time: 33.9 s (started: 2024-12-03 21:53:02 +00:00

## Run DiffDock

In [ ]:
#%cd /content/DiffDock
#!python -m inference --protein_ligand_csv /tmp/input_protein_ligand.csv --out_dir results/user_predictions_small --inference_steps 20 --samples_per_complex 40 --batch_size 10
%cd /content/DiffDock
!python -m inference --config default_inference_args.yaml  --protein_ligand_csv data/protein_ligand_example.csv --out_dir results/user_predictions_small

/content/DiffDock
Traceback (most recent call last):
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/content/DiffDock/inference.py", line 28, in <module>
    import utils.utils
  File "/content/DiffDock/utils/utils.py", line 17, in <module>
    from models.aa_model import AAModel
  File "/content/DiffDock/models/aa_model.py", line 13, in <module>
    from utils import so3, torus
  File "/content/DiffDock/utils/so3.py", line 54, in <module>
    _exp_vals = np.asarray([_expansion(_omegas_array, eps) for eps in _eps_array])
  File "/content/DiffDock/utils/so3.py", line 54, in <listcomp>
    _exp_vals = np.asarray([_expansion(_omegas_array, eps) for eps in _eps_array])
  File "/content/DiffDock/utils/so3.py", line 24, in _expansion
    * np.sin(omega * (l_vec + 1 / 2)) / np.sin(omega / 2)).sum(0)
KeyboardInterrupt
time: 2mi

In [ ]:
!python app/main.py

 navigate to http://localhost:7860 in your browser

## Download results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

if download_results:
    from google.colab import files
    from glob import glob
    from shlex import quote

    %cd /content/drive/MyDrive/DiffDock/
    out_fs = []
    for pdb_file in pdb_files:
        !cp {pdb_file} .
        pdb_file_root = pdb_file.split("/")[-1]
        out_fs.append(f"{pdb_file_root}")
        for smiles in smiless:
            sglob = ''.join([c if c in "CONH" else "?" for c in smiles])
            out_fs += glob(f"*{pdb_file_root}*{sglob}*/rank*_*.sdf")

    out_fs_bash = ' '.join([quote(f) for f in out_fs])
    !tar cvf diffdock_results.tar {out_fs_bash}
    files.download("diffdock_results.tar")

In [ ]:
import os
chemin_fichier_tar = "/content/drive/MyDrive/DiffDock/diffdock_results.tar"


In [ ]:
!tar -xvf "/content/drive/MyDrive/DiffDock/diffdock_results.tar" -C "/content/drive/MyDrive/DiffDock/"

In [ ]:
!pip install py3Dmol
import py3Dmol

In [ ]:
with open("/content/drive/MyDrive/DiffDock/6agt.pdb") as ifile:
    system = "".join([x for x in ifile])

In [ ]:
view = py3Dmol.view(width=400, height=300)
view.addModelsAsFrames(system)
view.setStyle({'model': -1}, {"cartoon": {'color': 'spectrum'}})
view.zoomTo()
view.show()

In [ ]:
with open("/content/drive/MyDrive/DiffDock/1a0q_protein_processed.pdb") as ifile:
    system2 = "".join([x for x in ifile])

In [ ]:
view = py3Dmol.view(width=400, height=300)
view.addModelsAsFrames(system2)
view.setStyle({'model': -1}, {"cartoon": {'color': 'spectrum'}})
view.zoomTo()
view.show()

In [ ]:
%cd /content/DiffDock/data
!wget https://zenodo.org/records/6408497/files/PDBBind.zip?download=1 -O PDBBind.zip
!unzip PDBBind.zip

Streaming output truncated to the last 5000 lines.
  inflating: PDBBind_processed/6h7z/6h7z_ligand.mol2  
  inflating: PDBBind_processed/6h7z/6h7z_ligand.sdf  
  inflating: PDBBind_processed/6h7z/6h7z_protein_processed.pdb  
   creating: PDBBind_processed/6h8c/
  inflating: PDBBind_processed/6h8c/6h8c_ligand.mol2  
  inflating: PDBBind_processed/6h8c/6h8c_ligand.sdf  
  inflating: PDBBind_processed/6h8c/6h8c_protein_processed.pdb  
   creating: PDBBind_processed/6h8s/
  inflating: PDBBind_processed/6h8s/6h8s_ligand.mol2  
  inflating: PDBBind_processed/6h8s/6h8s_ligand.sdf  
  inflating: PDBBind_processed/6h8s/6h8s_protein_processed.pdb  
   creating: PDBBind_processed/6h96/
  inflating: PDBBind_processed/6h96/6h96_ligand.mol2  
  inflating: PDBBind_processed/6h96/6h96_ligand.sdf  
  inflating: PDBBind_processed/6h96/6h96_protein_processed.pdb  
   creating: PDBBind_processed/6h9b/
  inflating: PDBBind_processed/6h9b/6h9b_ligand.mol2  
  inflating: PDBBind_processed/6h9b/6h9b_ligand.sd

1a50 ?
... .DSSTORE...
dir_path = os.path.join(data_dir, name)
if os.path.isdir(dir_path) and not os.listdir(dir_path):
    # Skip if it's an empty directory
    continue

In [ ]:
%cd /content/DiffDock/
!python datasets/esm_embedding_preparation.py --data_dir data/PDBBind_processed

/content/DiffDock
  0% 28/19120 [00:03<33:04,  9.62it/s]encountered unknown AA:  SEP  in the complex  data/PDBBind_processed/6mlw/6mlw_protein_processed.pdb . Replacing it with a dash - .
  0% 47/19120 [00:04<21:22, 14.87it/s]encountered unknown AA:  KCX  in the complex  data/PDBBind_processed/5qac/5qac_protein_processed.pdb . Replacing it with a dash - .
  0% 58/19120 [00:05<31:56,  9.95it/s]encountered unknown AA:  SEP  in the complex  data/PDBBind_processed/2uw8/2uw8_protein_processed.pdb . Replacing it with a dash - .
encountered unknown AA:  TPO  in the complex  data/PDBBind_processed/2uw8/2uw8_protein_processed.pdb . Replacing it with a dash - .
encountered unknown AA:  SEP  in the complex  data/PDBBind_processed/2uw8/2uw8_protein_processed.pdb . Replacing it with a dash - .
  0% 75/19120 [00:07<19:14, 16.49it/s]encountered unknown AA:  PTR  in the complex  data/PDBBind_processed/5amn/5amn_protein_processed.pdb . Replacing it with a dash - .
encountered unknown AA:  PTR  in the c

In [ ]:
!mv /content/DiffDock/data/prepared_for_esm.fasta /content/drive/My\ Drive/

mv: cannot move '/content/DiffDock/data/prepared_for_esm.fasta' to '/content/drive/My Drive/': No such file or directory
time: 106 ms (started: 2024-12-03 22:33:59 +00:00)


In [ ]:
%cd /content/DiffDock/esm
!python scripts/extract.py esm2_t33_650M_UR50D /content/DiffDock/data/prepared_for_esm.fasta  embeddings_output --repr_layers 33 --include per_tok --truncation_seq_length 4096

/content/DiffDock/esm
Transferred model to GPU
Read /content/DiffDock/data/prepared_for_esm.fasta with 24926 sequences
Processing 1 of 1809 batches (117 sequences)
Processing 2 of 1809 batches (85 sequences)
Processing 3 of 1809 batches (77 sequences)
Processing 4 of 1809 batches (70 sequences)
Processing 5 of 1809 batches (65 sequences)
Processing 6 of 1809 batches (60 sequences)
Processing 7 of 1809 batches (56 sequences)
Processing 8 of 1809 batches (52 sequences)
Processing 9 of 1809 batches (49 sequences)
Processing 10 of 1809 batches (48 sequences)
Processing 11 of 1809 batches (47 sequences)
Processing 12 of 1809 batches (46 sequences)
Processing 13 of 1809 batches (45 sequences)
Processing 14 of 1809 batches (45 sequences)
Processing 15 of 1809 batches (44 sequences)
Processing 16 of 1809 batches (43 sequences)
Processing 17 of 1809 batches (42 sequences)
Processing 18 of 1809 batches (42 sequences)
Processing 19 of 1809 batches (41 sequences)
Processing 20 of 1809 batches (41 

In [1]:
%cd /content/DiffDock/
!python datasets/esm_embeddings_to_pt.py

[Errno 2] No such file or directory: '/content/DiffDock/'
/content
python3: can't open file '/content/datasets/esm_embeddings_to_pt.py': [Errno 2] No such file or directory


In [ ]:

!python -m evaluate  --config default_inference_args.yaml --split_path data/splits/timesplit_test --split_path data/splits/timesplit_test --batch_size 10 --esm_embeddings_path data/esm2_embeddings.pt --data_dir data/PDBBind_processed/ --tqdm --split test --chain_cutoff 10 --dataset pdbbind